# Data Quality

Notebook demonstrates the reusable `DataQualityAgent` workflow:

- load collected text data or use a fallback demo dataframe
- detect missing values, duplicates, outliers, and class imbalance
- compare conservative and strict cleaning strategies
- export review artifacts for human approval


In [ ]:
from pathlib import Path

import pandas as pd

from agents.data_quality_agent import DataQualityAgent

project_root = Path.cwd()
collection_artifacts_dir = project_root / "artifacts" / "data_collection"
csv_candidates = sorted(collection_artifacts_dir.rglob("merged_dataframe.csv"))

if csv_candidates:
    df = pd.read_csv(csv_candidates[-1])
else:
    df = pd.DataFrame(
        {
            "prompt": [
                "Classify this review",
                "  classify   this review  ",
                "Summarize the article",
                None,
                "A" * 800,
            ],
            "text": [
                "Loved the movie.",
                "Loved the movie.",
                "Short summary request.",
                "   ",
                "Long noisy sample " * 80,
            ],
            "label": ["positive", "positive", "neutral", "negative", "negative"],
            "score": [1.0, 1.0, 2.0, None, 999.0],
        }
    )

agent = DataQualityAgent(
    config={
        "quality": {
            "project_root": str(project_root),
            "duplicate_subset": ["prompt"],
            "critical_text_columns": ["prompt", "text"],
        }
    }
)

df.head()


In [ ]:
report = agent.detect_issues(df)
report.to_dict()


In [ ]:
preview_strategies = agent.default_preview_strategies(df)
conservative_df = agent.fix(df, preview_strategies["conservative"])
strict_df = agent.fix(df, preview_strategies["strict"])

comparison_df = pd.DataFrame(
    {
        "metric": [item["metric"] for item in agent.compare(df, conservative_df).metrics],
        "conservative_after": [item["after"] for item in agent.compare(df, conservative_df).metrics],
        "strict_after": [item["after"] for item in agent.compare(df, strict_df).metrics],
    }
)
comparison_df


## Strategy Rationale

Preferred final strategy should be chosen only after human review of the exported bundle.

Example rationale template:

- conservative preview preserves borderline long-form NLP samples and only removes clearly broken rows
- strict preview is safer for annotation throughput but may remove rare yet valid text examples
- final decision must be copied into `review/quality_review_decision_template.json` with `approved=true`


In [ ]:
artifacts = agent.prepare_review_bundle(df, preview_strategies=preview_strategies)
artifacts


In [ ]:
from IPython.display import Image, display

for relative_path in [
    project_root / "reports/quality/plots/missing_values.png",
    project_root / "reports/quality/plots/duplicates.png",
    project_root / "reports/quality/plots/outliers.png",
    project_root / "reports/quality/plots/class_imbalance.png",
]:
    display(Image(filename=str(relative_path)))
